In [1]:
# Step 1 — Imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# Step 2 — Connect to Ollama
# Ollama exposes an OpenAI-compatible API at localhost:11434
# The api_key value is required by the SDK but ignored by Ollama

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL = "llama3.2"

In [3]:
# Step 3 — Quick connection test

response = ollama.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello\! Reply with one sentence only."}]
)
print(response.choices[0].message.content)

<>:5: SyntaxWarning: invalid escape sequence '\!'
<>:5: SyntaxWarning: invalid escape sequence '\!'
/var/folders/32/fdngh_l52yd4cpmdpm6c4_000000gn/T/ipykernel_28551/2935225330.py:5: SyntaxWarning: invalid escape sequence '\!'
  messages=[{"role": "user", "content": "Hello\! Reply with one sentence only."}]


I'm happy to assist you.


In [4]:
# Step 4 — Website scraper helper
# Fetches and cleans the text content of any URL

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for tag in soup.body(["script", "style", "img", "input"]):
            tag.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [5]:
# Step 5 — Try the scraper

content = fetch_website_contents("https://edwarddonner.com")
print(content)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190

In [6]:
# Step 6 — Define the system and user prompts

system_prompt = """
You are a snarky assistant that analyzes the contents of a website
and provides a short, snarky, humorous summary, ignoring navigation-related text.
Respond in markdown. Do not wrap the markdown in a code block.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""

In [7]:
# Step 7 — Build the messages list
# OpenAI-compatible format: system prompt + user prompt

def messages_for(website_content):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]

# Preview the message structure
messages_for("Example website content here")

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website\nand provides a short, snarky, humorous summary, ignoring navigation-related text.\nRespond in markdown. Do not wrap the markdown in a code block.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, summarize these too.\n\nExample website content here'}]

In [8]:
# Step 8 — summarize(): fetches a URL and calls Ollama

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [10]:
# Step 9 — display_summary(): renders the result as formatted Markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [11]:
# Step 10 — Run it\! Summarize CNN

display_summary("https://cnn.com")

**Summary**: CNN's website is a news hub with various sections, including Breaking News, Politics, Business, Entertainment, and more. It also offers videos, live TV, and podcasts. Yay, another online news site.

**News and Announcements**

* The website asks for user feedback on ads, which can be submitted via a handy form. Because who doesn't love giving constructive criticism?
* CNN has made it clear that *everything* on the site is "Breaking News," which kind of defeats the purpose.
* A dedicated section exists for submitting feedback on ads, making it easy to let your thoughts be heard (or annoy others with them).
* Various sub-sections cover topics like politics, business, and health, but they're all just "News" in their respective categories. You'd think there would be more depth or categorization, but nope.
* Unfortunately, the website asks you to log in for even basic functions, which is a real bummer after spending all that time on the ad feedback form.

In [12]:
display_summary("https://anthropic.com")

This website appears to be about Anthropic, a public benefit corporation that aims to secure the benefits and mitigate the risks of AI. They've created an AI platform called Claude, which offers a space for genuine conversations without ads or sponsored content.

News:

* 81,000 people participated in the largest study ever done on AI, and its findings are supposedly summarized somewhere on this site (but not found among these contents).
* Anthropic released version 4.6 of their AI model, Claude Opus, on February 5, 2026.
* There's a recent announcement or two about new releases of the Claude platform, but I couldn't dig up the content since it was buried under annoying navigation menus and other non-essential text.

That's basically it – more of a corporate update than actual news.

In [13]:
display_summary("https://edwarddonner.com")

Ed's website is basically an intro to his life, interests, and work. He's a guy who likes writing code, playing with large language models (LLMs), making music, and... also making more code, because why not? On the 'About' page, he talks about co-founding Nebula.io, his previous start-up untapt getting acquired in 2021, and how much joy it brings him to share LLM info with anyone who'll listen (he even convinced others to make some Udemy courses which did surprisingly well).

He also shares some recent updates like new resources on AI development and sharing them as news-like entries.